# 12.1 音訊是什麼 tensor？


一根琴弦振動，空氣壓力隨時間上下起伏。數位音訊把這種起伏按固定時間間隔記成一串數字，叫波形waveform。一次起伏有多大是振幅，起伏重複有多快是頻率。音比較高不代表振幅一定更大；本節先認清模型最前面收到哪一串數字。

前置是[W.3的一維tensor](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。單聲道有N個時間樣本，形狀[N]；一批B段等長單聲道可為[B,N]。左右雙聲道則需要另外一條通道軸，不能把左右兩串接成一段兩倍長的錄音，否則時間意義會變。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.multimodal import tone

wave = tone(440.0)
print("樣本形狀", tuple(wave.shape))
print("前三個值", [round(v, 3) for v in wave[:3].tolist()])
print("最小最大", round(wave.min().item(), 2), round(wave.max().item(), 2))

輸入`tone(440.0)`生成440Hz的正弦單音，Hz表示每秒重複幾次。正弦是一種特定的平滑曲線；在這個例子裡，它規律地從0升到最高、回到0、降到最低、再回到0，這完整一輪叫一個週期。440Hz就是每秒走完440輪。工具預設長0.1秒、每秒記錄16000個樣本，振幅0.5。它不讀取檔案或播放聲音，直接算出浮點tensor。輸出形狀`(1600,)`，前三值約[0.0,0.086,0.169]，最小最大約-0.5與0.5。波形先從0向上，之後會繼續上下往復；負數不是錯誤，而是相對基準的另一方向起伏。

每個數字本身沒有附時間，必須另外知道取樣率16000，才能說第1個索引在1/16000秒。1600是樣本數量，不是1600Hz，也不是1600個語言token。進入音訊編碼器之前還會切成短時間框，每框形成一條特徵，這個轉換會在後面逐步解釋。

振幅是這份數字記錄的尺度，常與聲音強弱有關，但真實聽感還受設備與頻率影響；不要把0.5直接叫固定分貝。頻率440控制起伏速度，而工具裡的0.5控制上下幅度，兩者在這個生成器裡能獨立改變。

若把波形畫成圖，橫軸是時間或樣本索引，縱軸是振幅；曲線越密集通常表示起伏越快，而不是樣本編號越大代表聲音越高。下圖只放大一個週期，用「這一輪走完多少」作橫軸：走到1/4輪時振幅是0.5，1/2輪時回到0，3/4輪時是-0.5，走完一輪又回到0，接著重複。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_wave_cycle.svg")))

相位指起伏目前走到週期中的哪個位置，可以先當作「這一輪的進度」。相位不是振幅：同樣振幅0，可能是在起點向上，也可能是在半輪時向下。工具從圖左的0開始，索引0是第一點；索引1是1/16000秒後的第二點，這時440Hz的波走了440/16000輪，還沒到最高點，生成器算出的振幅約0.086。要解釋的是取樣時刻在曲線上的哪裡，不需要先學正弦公式。

這條曲線是確定生成的測試輸入，方便我們知道答案，不是從音樂中成功抽取了音高。真實錄音還可能有噪聲、靜音或多個聲源，單看前三個值無法辨識全部內容；後面才會在短框內匯整這些變化。

練習只把440改880，先預測形狀仍1600、極值仍約±0.5，而前幾個值變得更快上升，再執行核對。它在同樣時間內起伏約兩倍次數，不會因此多出兩倍樣本。下一節將解釋樣本數量如何與真實時間聯繫。
